# Sentiment Analysis: Synthetic Data Generation for Kaggle

## Fast Multi-Model Conversation Generation - Kaggle GPU Optimized

This notebook generates synthetic conversations using **FREE** open-source LLMs optimized for Kaggle's GPU environment.

**Key Features:**
- ✅ **GPU-Accelerated** - Uses Kaggle's free GPU (CUDA)
- ✅ Uses Ollama for fast inference
- ✅ No additional cost - completely FREE
- ✅ Kaggle dataset export ready  
- ✅ Unlimited runtime (no timeout)
- ✅ VADER sentiment analysis included

**Quick Start:**
1. Enable GPU in notebook settings (if not already enabled)
2. Run all cells (faster with GPU acceleration)
3. Download results from `/kaggle/working/`
4. Export to Kaggle dataset if desired

In [ ]:
import os
import sys
import subprocess
import json
import time
import requests
import logging
from datetime import datetime
from typing import Dict, Tuple
import random

# GPU Detection and Configuration
try:
    import torch
    USE_GPU = torch.cuda.is_available()
    GPU_COUNT = torch.cuda.device_count()
    GPU_NAME = torch.cuda.get_device_name(0) if USE_GPU else "None"
    CUDA_VERSION = torch.version.cuda
except ImportError:
    USE_GPU = False
    GPU_COUNT = 0
    GPU_NAME = "N/A"
    CUDA_VERSION = "N/A"

# Kaggle-specific setup
IS_KAGGLE = os.path.exists('/kaggle/working')
OUTPUT_DIR = '/kaggle/working/synthetic_conversations'
os.makedirs(OUTPUT_DIR, exist_ok=True)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print(f"📍 Running on Kaggle")
print(f"📁 Output directory: {OUTPUT_DIR}")
print(f"\n🔧 GPU Configuration:")
print(f"   CUDA Available: {'✅ YES' if USE_GPU else '❌ NO'}")
if USE_GPU:
    print(f"   GPU Count: {GPU_COUNT}")
    print(f"   GPU Device: {GPU_NAME}")
    print(f"   CUDA Version: {CUDA_VERSION}")
    print(f"\n✨ GPU acceleration ENABLED for faster processing!")
else:
    print(f"   ⚠️  Running on CPU - Consider enabling GPU in notebook settings")

# Install dependencies
print("\n📦 Installing dependencies...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "requests", "nltk", "tqdm", "psutil"])
print("✅ Dependencies ready")

In [ ]:
import subprocess
import time
import threading
import requests
import os

print("📦 Installing Ollama...")
ollama_executable_path = "/usr/local/bin/ollama"  # Expected installation path

# Configure GPU environment variables
if USE_GPU:
    os.environ['CUDA_VISIBLE_DEVICES'] = '0'
    os.environ['OLLAMA_GPU_LAYERS'] = '99'
    os.environ['OLLAMA_NUM_GPU'] = '1'
    print("🔧 GPU environment configured for Ollama")
else:
    print("⚠️  GPU not detected - will run on CPU")

try:
    # Install zstd dependency as requested by the Ollama installer
    print("Installing zstd dependency...")
    subprocess.run(
        ["sudo", "apt-get", "update"],
        capture_output=True, text=True, check=True
    )
    subprocess.run(
        ["sudo", "apt-get", "install", "-y", "zstd"],
        capture_output=True, text=True, check=True
    )
    print("✅ zstd installed successfully.")

    # Correctly execute the Ollama installation script by piping curl output to sh
    print("Running Ollama installation script...")
    install_command = "curl -fsSL https://ollama.ai/install.sh | sh"
    install_result = subprocess.run(
        install_command,
        shell=True,
        capture_output=True,
        text=True,
        check=True
    )
    print("✅ Ollama installation script executed successfully.")
    if install_result.stdout:
        print("Installer Output (stdout):\n", install_result.stdout)
    if install_result.stderr:
        print("Installer Output (stderr):\n", install_result.stderr)

    # Verify if the ollama executable exists after installation
    if not os.path.exists(ollama_executable_path):
        raise FileNotFoundError(f"Ollama executable not found at {ollama_executable_path} after installation.")
    print(f"✅ Ollama executable verified at: {ollama_executable_path}")

except subprocess.CalledProcessError as e:
    print(f"❌ A subprocess command failed with return code {e.returncode}.")
    print("Command:", e.cmd)
    print("Stdout:", e.stdout)
    print("Stderr:", e.stderr)
    print("Please check the output for errors during installation or dependency setup.")
    raise
except FileNotFoundError as e:
    print(f"❌ Error: {e}")
    raise
except Exception as e:
    print(f"❌ An unexpected error occurred during Ollama installation: {e}")
    raise


# Start Ollama in background with GPU support
print("\n🚀 Starting Ollama service...")
def run_ollama():
    env = os.environ.copy()
    if USE_GPU:
        env['CUDA_VISIBLE_DEVICES'] = '0'
        env['OLLAMA_GPU_LAYERS'] = '99'
        env['OLLAMA_NUM_GPU'] = '1'
    
    subprocess.Popen(
        [ollama_executable_path, "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        env=env
    )

# Start Ollama in thread
ollama_thread = threading.Thread(target=run_ollama, daemon=True)
ollama_thread.start()

# Wait for Ollama to start
ollama_started = False
for i in range(30):
    try:
        response = requests.get("http://localhost:11434/api/tags", timeout=2)
        if response.status_code == 200:
            print("✅ Ollama is running")
            if USE_GPU:
                print("   GPU acceleration enabled 🚀")
            ollama_started = True
            break
    except requests.exceptions.ConnectionError:
        pass
    except Exception as e:
        print(f"⚠️ Error checking Ollama status: {e}")
    time.sleep(1)
    if i % 5 == 0:
        print(f"⏳ Waiting for Ollama... ({i+1}s)")

if not ollama_started:
    print("❌ Ollama service did not start within the expected time. Please check logs for errors.")
    raise RuntimeError("Ollama service failed to start.")


print("\n📥 Downloading models (this takes ~10-20 minutes on first run)...")
models = ["llama2", "mistral", "neural-chat", "phi"]

for model in models:
    print(f"\n⬇️  Downloading {model}...")
    result = subprocess.run([ollama_executable_path, "pull", model],
                          capture_output=True, text=True)
    if "success" in result.stdout.lower() or "latest" in result.stdout.lower():
        print(f"✅ {model} ready")
    else:
        print(f"📥 {model} downloading in background (or failed, check stderr)...")
        if result.stdout:
            print(f"Stdout for {model}:\n", result.stdout)
        if result.stderr:
            print(f"Stderr for {model}:\n", result.stderr)

print("\n✅ All models ready for generation!")
if USE_GPU:
    print("   Running with GPU acceleration ⚡")

In [ ]:
class ConversationGenerator:
    def __init__(self):
        self.topics = [
        "rape",
        "climate change",
        "depression",
        "gambling",
        "drug addiction",
        "How AI agents act, respond, and execute strategies"
        "How agents maintain identity, cold-start costs"
        ]
    
    def call_ollama(self, model: str, prompt: str, timeout: int = 180) -> str:
        try:
            response = requests.post(
                "http://localhost:11434/api/generate",
                json={"model": model, "prompt": prompt, "stream": False},
                timeout=timeout
            )
            return response.json().get("response", "").strip()[:200]
        except Exception as e:
            logger.warning(f"Ollama error: {e}")
            return ""
    
    def generate(self, model_a: str, model_b: str, num_conversations: int = 50):
        conversations = []
        
        for i in range(num_conversations):
            topic = random.choice(self.topics)
            
            # Model A initiates
            msg_a = self.call_ollama(model_a, f"Briefly discuss: {topic}")
            if not msg_a:
                continue
            
            time.sleep(0.2)
            
            # Model B responds
            msg_b = self.call_ollama(model_b, f"Respond to: {msg_a}\n\nKeep it brief.")
            if not msg_b:
                continue
            
            conversations.append({
                "timestamp": datetime.now().isoformat(),
                "pair": f"{model_a}-{model_b}",
                "topic": topic,
                "messages": [
                    {"speaker": model_a, "text": msg_a},
                    {"speaker": model_b, "text": msg_b}
                ]
            })
        
        return conversations

generator = ConversationGenerator()
print("✅ Generator initialized")

In [ ]:
from tqdm import tqdm

# Configuration
CONFIG = {
    "pairs": [
        ("llama2", "mistral"),
        ("mistral", "phi"),
        ("phi", "llama2"),
        ("llama2", "phi"),
        ("mistral", "llama2"),
        ("phi", "mistral"),
    ],
    "conversations_per_pair": 500,  # 500 conversations per pair = 3000 total
}

print("⚙️  Configuration:")
print(f"  Model pairs: {len(CONFIG['pairs'])}")
print(f"  Conversations per pair: {CONFIG['conversations_per_pair']}")
print(f"  Total conversations: {len(CONFIG['pairs']) * CONFIG['conversations_per_pair']}")

# Generate conversations
print("\n🚀 GENERATING CONVERSATIONS\n")

all_conversations = []
start_time = time.time()

for pair_idx, (model_a, model_b) in enumerate(CONFIG['pairs'], 1):
    print(f"📍 Pair {pair_idx}/{len(CONFIG['pairs'])}: {model_a} ↔ {model_b}")
    
    conversations = generator.generate(model_a, model_b, CONFIG['conversations_per_pair'])
    all_conversations.extend(conversations)
    
    print(f"   ✅ Generated {len(conversations)} conversations")

elapsed = time.time() - start_time

# Save conversations
output_file = os.path.join(OUTPUT_DIR, 'conversations.jsonl')
with open(output_file, 'w') as f:
    for conv in all_conversations:
        f.write(json.dumps(conv) + '\n')

print(f"\n{'='*60}")
print(f"✅ GENERATION COMPLETE")
print(f"{'='*60}")
print(f"Total conversations: {len(all_conversations)}")
print(f"Time: {elapsed:.1f}s ({elapsed/60:.1f} min)")
print(f"Rate: {len(all_conversations)/(elapsed/60):.1f} convos/min")
print(f"Saved to: {output_file}")

In [ ]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

# Setup VADER
try:
    nltk.data.find('vader_lexicon')
except LookupError:
    nltk.download('vader_lexicon', quiet=True)

sia = SentimentIntensityAnalyzer()

print("🔍 Running sentiment analysis...\n")

analysis_file = os.path.join(OUTPUT_DIR, 'sentiment_analysis.jsonl')
sentiments = {"positive": 0, "neutral": 0, "negative": 0}

with open(output_file, 'r') as infile, open(analysis_file, 'w') as outfile:
    for line in tqdm(infile, desc="Analyzing", total=len(all_conversations)):
        conv = json.loads(line)
        
        for msg in conv['messages']:
            scores = sia.polarity_scores(msg['text'])
            compound = scores['compound']
            
            if compound > 0.05:
                label = "positive"
                sentiments["positive"] += 1
            elif compound < -0.05:
                label = "negative"
                sentiments["negative"] += 1
            else:
                label = "neutral"
                sentiments["neutral"] += 1
            
            analysis = {
                "timestamp": conv['timestamp'],
                "model": msg['speaker'],
                "text": msg['text'],
                "sentiment": label,
                "scores": {
                    "positive": round(scores['pos'], 3),
                    "neutral": round(scores['neu'], 3),
                    "negative": round(scores['neg'], 3),
                    "compound": round(scores['compound'], 3)
                }
            }
            outfile.write(json.dumps(analysis) + '\n')

print(f"\n📊 Sentiment Distribution:")
total = sum(sentiments.values())
for sent, count in sentiments.items():
    pct = (count / total * 100) if total > 0 else 0
    print(f"  {sent.capitalize():8s}: {count:4d} ({pct:5.1f}%)")

print(f"\n✅ Analysis saved to: {analysis_file}")

In [ ]:
import psutil

print("\n" + "="*70)
print("📊 FINAL SUMMARY & RESULTS")
print("="*70)

# File info
conv_size = os.path.getsize(output_file) / 1024 / 1024
analysis_size = os.path.getsize(analysis_file) / 1024 / 1024

print(f"\n📁 Output Files (in /kaggle/working/synthetic_conversations/):")
print(f"  • conversations.jsonl: {conv_size:.2f} MB")
print(f"  • sentiment_analysis.jsonl: {analysis_size:.2f} MB")
print(f"  • Total: {conv_size + analysis_size:.2f} MB")

# Performance
print(f"\n⚡ Performance Metrics:")
print(f"  CPU Cores: {psutil.cpu_count()}")
print(f"  Total Memory: {psutil.virtual_memory().total / 1024**3:.1f} GB")
print(f"  GPU Acceleration: {'✅ ENABLED' if USE_GPU else '❌ Not Available'}")
if USE_GPU:
    print(f"  GPU Device: {GPU_NAME}")
    print(f"  CUDA Version: {CUDA_VERSION}")
print(f"  Total Time: {elapsed:.1f}s")

# Cost
print(f"\n💰 Cost Analysis:")
print(f"  Your cost: $0.00 (FREE! 🎉)")
print(f"  vs OpenAI API: ~${len(all_conversations) * 0.5 * 0.01 / 1000:.2f}")

print(f"\n📝 Next Steps:")
print(f"  1. Find files in /kaggle/working/synthetic_conversations/")
print(f"  2. Download both .jsonl files")
print(f"  3. Use for training your sentiment models!")

print(f"\n" + "="*70)
print("✅ Pipeline Complete - GPU-Accelerated Processing Done!")
print("="*70)